# Exercise 19 - Bitcoin values

In this exercise, I want you to retrieve the dates and values for Bitcoin over the most recent year as of when you read this. (For that reason, your results will look different from mine, even if you use the same code.) Once you have retrieved this data, I want you to produce a report showing:

* The closing price for the most recent trading day
* The lowest historical price and the date of that price
* The highest historical price and the date of that price

As of this writing, you can retrieve Bitcoin’s price history in CSV format at https://api.blockchain.info/charts/market-price?format=csv.

**NOTE** Many stock-history sites require that you register and log in before retrieving data, but as of this writing, the URL I provided here does not.

In [34]:
import pandas as pd


bitcoin = pd.read_csv(
    filepath_or_buffer="https://api.blockchain.info/charts/market-price?format=csv",
    header=None,
    names=["date", "price"],
)

bitcoin.head(10)

,date,price
0,2025-05-23 00:00:00,111722.19
1,2025-05-24 00:00:00,107303.71
2,2025-05-25 00:00:00,107809.79
3,2025-05-26 00:00:00,109029.24
4,2025-05-27 00:00:00,109481.32
5,2025-05-28 00:00:00,108987.67
6,2025-05-29 00:00:00,107808.38
7,2025-05-30 00:00:00,105588.16
8,2025-05-31 00:00:00,104027.67
9,2025-06-01 00:00:00,104655.37


* The closing price for the most recent trading day.

In [35]:
closing_price = bitcoin.iloc[-1]

print(f"closing_price for the most recent day:\ndate: {closing_price['date']}\nprice: ${closing_price['price']:,.2f}")

closing_price for the most recent day:
date: 2026-05-23 00:00:00
price: $75,460.93


* The lowest historical price and the date of that price.

In [36]:
lowest_price = bitcoin.iloc[bitcoin["price"].idxmin()]

print(f"lowest price:\ndate: {lowest_price['date']}\nprice: ${lowest_price['price']:,.2f}")

lowest price:
date: 2026-02-06 00:00:00
price: $62,812.06


* The highest historical price and the date of that price.

In [37]:
highest_price = bitcoin.iloc[bitcoin["price"].idxmax()]

print(f"highest price:\ndate: {highest_price['date']}\nprice: ${highest_price['price']:,.2f}")

highest price:
date: 2025-10-07 00:00:00
price: $124,776.68


## Beyond the exercise

* In this exercise, you downloaded the information into a data frame and then performed calculations on it. Without assigning the downloaded data to an interim variable, can you return the current value? Your solution should consist of a single line of code that includes the download, selection, and calculation.

In [38]:
pd.read_csv(
    filepath_or_buffer="https://api.blockchain.info/charts/market-price?format=csv",
    header=None,
    names=["date", "price"],
).iloc[-1]["price"]

np.float64(75460.93)

* The ```pd.read_html``` function, like ```pd.read_csv```, takes a file-like object or a URL. It assumes that it will encounter HTML-formatted text containing at least one table. It turns each table into a data frame and then returns a list of those data frames. With this in mind, retrieve one year of historical S&P 500 data from Yahoo Finance (https://finance.yahoo.com/quote/%5EGSPC/history?p=%5EGSPC), looking only at the ```Date```, ```Close```, and ```Volume``` columns. Show the date and volume of the days with the highest and lowest ```Close``` values. Note that Yahoo seems to look at the ```User-Agent``` header in the HTTP request, which cannot be set in ```read_html```. So you’ll need to use ```requests``` to retrieve the data, setting ```User-Agent``` to a string equal to '```Mozilla 5.0```'. Turn the content of the result into a ```StringIO```, and then feed that to ```read_html``` and retrieve the data.

In [39]:
import requests
from io import StringIO

response = requests.get(
    "https://finance.yahoo.com/quote/%5EGSPC/history?p=%5EGSPC",
    headers = {# without these headers can't retrieve data
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Upgrade-Insecure-Requests": "1"
    }
)

string_io = StringIO(response.content.decode())

sp_500 = pd.read_html(# this function return a list of dataframes
    io=string_io,
    index_col="Date",

)[0] # first dataframe from the list

sp_500 = sp_500[["Close Close price adjusted for splits.", "Volume"]] # get the columns of interest from the original dataframe
sp_500.rename(columns={"Close Close price adjusted for splits.": "Close"}, inplace=True) # rename column

In [50]:
# show 10 data points with format according to each column
sp_500.head(10).style.format({
    "Close": '${:,.2f}',
    "Volume": '{:,.0f}',
})


,Close,Volume
Date,,
"May 22, 2026","$7,473.47","2,693,030,000"
"May 21, 2026","$7,445.72","5,440,620,000"
"May 20, 2026","$7,432.97","5,384,150,000"
"May 19, 2026","$7,353.61","5,441,140,000"
"May 18, 2026","$7,403.05","5,489,500,000"
"May 15, 2026","$7,408.50","5,582,070,000"
"May 14, 2026","$7,501.24","5,267,050,000"
"May 13, 2026","$7,444.25","5,716,600,000"
"May 12, 2026","$7,400.96","5,624,790,000"


In [60]:
df = sp_500.loc[sp_500["Close"].agg(["idxmin", "idxmax"])]

print(df.to_csv())

Date,Close,Volume
"May 23, 2025",5802.82,4662820000
"May 14, 2026",7501.24,5267050000

